# Querying Time Series Data

All read patterns for overlapping forecast series: latest values, valid-time and knowledge-time range filters, the full version history, and `read_relative` for window-based day-ahead queries.

In [1]:
try:
    import google.colab
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/rebase-energy/timedb/main/examples/colab_setup.py',
        '/tmp/colab_setup.py'
    )
    exec(open('/tmp/colab_setup.py').read())
except ImportError:
    pass

In [2]:
from datetime import datetime, timezone, timedelta, time
import polars as pl
import timedb as td

# valid_times span 2025-01-03 to 2025-01-05
forecast_base = datetime(2025, 1, 3, tzinfo=timezone.utc)
valid_times = [forecast_base + timedelta(hours=i) for i in range(48)]

## Setup

Three forecast runs at increasing knowledge_times over the same valid window.

In [3]:
td.delete()
td.create()

td.create_series("wind_forecast", unit="MW", labels={"site": "Gotland"}, overlapping=True)
td.create_series("solar_forecast", unit="MW", labels={"site": "Gotland"}, overlapping=True)

# Three forecast runs for wind_forecast
runs = [
    (forecast_base - timedelta(days=2), [4.0 + i * 0.10 for i in range(48)]),
    (forecast_base - timedelta(days=1), [5.0 + i * 0.15 for i in range(48)]),
    (forecast_base,                     [5.5 + i * 0.12 for i in range(48)]),
]

col = td.get_series("wind_forecast").where(site="Gotland")
for kt, values in runs:
    df = pl.DataFrame({
        "knowledge_time": pl.Series([kt] * 48).cast(pl.Datetime("us", "UTC")),
        "valid_time":     pl.Series(valid_times).cast(pl.Datetime("us", "UTC")),
        "value":          values,
    })
    col.insert(df)

# One run for solar_forecast
td.get_series("solar_forecast").where(site="Gotland").insert(pl.DataFrame({
    "valid_time": pl.Series(valid_times).cast(pl.Datetime("us", "UTC")),
    "value": [2.0 + i * 0.04 for i in range(48)],
}))

print("✓ Setup complete")

Creating database schema...
  ✓ PostgreSQL: series_table
  ✓ ClickHouse: runs_table, flat, overlapping_short/medium/long
✓ Schema created successfully


✓ Setup complete


## Latest Values

`read()` returns the latest value per `valid_time` — the most recently issued forecast.

In [4]:
ts = col.read()
print(f"✓ {ts.num_rows} rows (one per valid_time)")
print(ts)

✓ 48 rows (one per valid_time)
TimeSeries
┌─────────────────────────────────────────┐
│  Name:             wind_forecast        │
│  Shape:            SIMPLE               │
│  Rows:             48                   │
│  Timezone:         UTC                  │
│  Unit:             MW                   │
│  Timeseries type:  OVERLAPPING          │
│  Labels:           {'site': 'Gotland'}  │
├─────────────────────────────────────────┤
│                    wind_forecast        │
│  2025-01-03 00:00            5.5        │
│  2025-01-03 01:00           5.62        │
│  2025-01-03 02:00           5.74        │
│  ...                         ...        │
│  2025-01-04 21:00           10.9        │
│  2025-01-04 22:00          11.02        │
│  2025-01-04 23:00          11.14        │
└─────────────────────────────────────────┘


## Time Range Filter

`start_valid` / `end_valid` restrict which `valid_time` values are returned. `start_valid` is inclusive, `end_valid` is exclusive (half-open interval).

In [5]:
ts = col.read(
    start_valid=forecast_base,
    end_valid=forecast_base + timedelta(hours=24),
)
print(f"✓ {ts.num_rows} rows (first 24h of valid window)")

✓ 24 rows (first 24h of valid window)


## Specific Forecast Run

Use `start_known` / `end_known` to pin to the exact knowledge_time of a single run.

In [6]:
kt_run2 = forecast_base - timedelta(days=1)

ts = col.read(
    start_known=kt_run2,
    end_known=kt_run2 + timedelta(seconds=1),
)
print(f"✓ Run 2 only: {ts.num_rows} rows (knowledge_time={kt_run2})")

✓ Run 2 only: 0 rows (knowledge_time=2025-01-02 00:00:00+00:00)


## All Forecast Runs

`read(overlapping=True)` returns a multi-index `(knowledge_time, valid_time)` — one row per run × valid_time.

In [7]:
ts = col.read(overlapping=True)
df = ts.to_polars()
print(f"✓ {ts.num_rows} rows across {df['knowledge_time'].n_unique()} runs")
print(df.head())

✓ 144 rows across 3 runs
shape: (5, 3)
┌─────────────────────────┬─────────────────────────┬───────┐
│ knowledge_time          ┆ valid_time              ┆ value │
│ ---                     ┆ ---                     ┆ ---   │
│ datetime[μs, UTC]       ┆ datetime[μs, UTC]       ┆ f64   │
╞═════════════════════════╪═════════════════════════╪═══════╡
│ 2025-01-01 00:00:00 UTC ┆ 2025-01-03 00:00:00 UTC ┆ 4.0   │
│ 2025-01-01 00:00:00 UTC ┆ 2025-01-03 01:00:00 UTC ┆ 4.1   │
│ 2025-01-01 00:00:00 UTC ┆ 2025-01-03 02:00:00 UTC ┆ 4.2   │
│ 2025-01-01 00:00:00 UTC ┆ 2025-01-03 03:00:00 UTC ┆ 4.3   │
│ 2025-01-01 00:00:00 UTC ┆ 2025-01-03 04:00:00 UTC ┆ 4.4   │
└─────────────────────────┴─────────────────────────┴───────┘


## Day-Ahead Forecasts

`read_relative` selects the latest forecast issued at least N days before each valid day. Here: issued by noon the day before (`time_of_day=time(hour=12)`).

In [8]:
ts = col.read_relative(
    days_ahead=1,
    time_of_day=time(hour=12),
    start_valid=forecast_base,
)
print(f"✓ {ts.num_rows} day-ahead values")
print(ts)

✓ 47 day-ahead values
TimeSeries
┌─────────────────────────────────────────┐
│  Name:             wind_forecast        │
│  Shape:            SIMPLE               │
│  Rows:             47                   │
│  Timezone:         UTC                  │
│  Unit:             MW                   │
│  Timeseries type:  OVERLAPPING          │
│  Labels:           {'site': 'Gotland'}  │
├─────────────────────────────────────────┤
│                    wind_forecast        │
│  2025-01-03 01:00           5.15        │
│  2025-01-03 02:00            5.3        │
│  2025-01-03 03:00           5.45        │
│  ...                         ...        │
│  2025-01-04 21:00           10.9        │
│  2025-01-04 22:00          11.02        │
│  2025-01-04 23:00          11.14        │
└─────────────────────────────────────────┘


## Multiple Series

Read several series and join into a single DataFrame.

In [9]:
wind = td.get_series("wind_forecast").where(site="Gotland").read().to_polars()
solar = td.get_series("solar_forecast").where(site="Gotland").read().to_polars()

combined = wind.rename({"value": "wind_MW"}).join(
    solar.rename({"value": "solar_MW"}), on="valid_time"
)
print(combined.head())

shape: (5, 3)
┌─────────────────────────┬─────────┬──────────┐
│ valid_time              ┆ wind_MW ┆ solar_MW │
│ ---                     ┆ ---     ┆ ---      │
│ datetime[μs, UTC]       ┆ f64     ┆ f64      │
╞═════════════════════════╪═════════╪══════════╡
│ 2025-01-03 00:00:00 UTC ┆ 5.5     ┆ 2.0      │
│ 2025-01-03 01:00:00 UTC ┆ 5.62    ┆ 2.04     │
│ 2025-01-03 02:00:00 UTC ┆ 5.74    ┆ 2.08     │
│ 2025-01-03 03:00:00 UTC ┆ 5.86    ┆ 2.12     │
│ 2025-01-03 04:00:00 UTC ┆ 5.98    ┆ 2.16     │
└─────────────────────────┴─────────┴──────────┘


## Summary

| Pattern | Call |
|---------|------|
| Latest per valid_time | `read()` |
| Slice by valid window | `read(start_valid=..., end_valid=...)` |
| Single forecast run | `read(start_known=kt, end_known=kt + timedelta(seconds=1))` |
| Full revision history | `read(overlapping=True)` |
| Day-ahead window | `read_relative(days_ahead=N, time_of_day=...)` |